In [1]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

! wget $PREFIX/01-agentic-rag/code/ingest.py
! wget $PREFIX/01-agentic-rag/code/rag_helper.py
! wget $PREFIX/04-evaluation/code/evaluation_utils.py

--2026-07-10 13:54:06--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py.1’

ingest.py.1         100%[===================>]     738  --.-KB/s    in 0s      

2026-07-10 13:54:07 (17.3 MB/s) - ‘ingest.py.1’ saved [738/738]

--2026-07-10 13:54:07--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 

In [2]:
"""
Loading FAQ data
"""

from ingest import load_faq_data
documents = load_faq_data()

In [3]:
# generating questions only for LLM Zoomcamp course:

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [4]:
# renaming:

documents = documents_llm

In [5]:
# each document already has an ID field:

doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
"""
Generating questions with structured output
We want the output as a list of strings, so we define that structure with a Pydantic model:
"""

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
# instrutions to LLM:

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
"""
We ask the LLM to use different wording from the original document. 
This makes the evaluation more realistic - real users won't phrase their questions the same way as the FAQ.
"""

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
# preparing the document as json:

import json

user_prompt = json.dumps(doc)

In [10]:
# creating the msg:

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
"""
Until now we called responses.create and read response.output_text. 
For structured output we switch to responses.parse and pass text_format=Questions, which tells the API to return our class instead of free text.
"""

# calling the model:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
# accessing the parsed object:

result = response.output_parsed

print(result)

questions=['I just found this course — is it too late to join now?', 'Can I still start the course even if I missed the beginning?', 'If I join late, will I still be able to get a certificate?', 'Is there a deadline for submitting the project if I want the certificate?', "What do I need to do to qualify for the course certificate if I'm joining now?"]


In [13]:
# accessing the list directly:

print(result.questions)

['I just found this course — is it too late to join now?', 'Can I still start the course even if I missed the beginning?', 'If I join late, will I still be able to get a certificate?', 'Is there a deadline for submitting the project if I want the certificate?', "What do I need to do to qualify for the course certificate if I'm joining now?"]


In [14]:
"""
Reusable utilities:
putting it in a reusable helper
"""

from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I found it late, or is it too late now?', 'If I join after the course has already started, can I still get a certificate?', 'What do I need to do to qualify for the course certificate if I’m joining late?', 'Is it okay to enroll in the course now, even though I just heard about it?', 'Does late enrollment affect whether I can submit the project and get certified?']


In [15]:
"""
Tracking cost
"""

usage.input_tokens, usage.output_tokens


(207, 102)

In [16]:
from evaluation_utils import calc_price

# calculate the cost of this call:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000459, 'total_cost': 0.00061425}

In [17]:
# Now converting these questions into ground truth records:

records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late, or is it too late now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the course certificate if I’m joining late?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to enroll in the course now, even though I just heard about it?',
  'document': '74eb249bbf'},
 {'question': 'Does late enrollment affect whether I can submit the project and get certified?',
  'document': '74eb249bbf'}]

In [18]:
"""
Generating Ground Truth for All Documents.
For each document, we:

convert the document to JSON so we can send it to the LLM
ask the LLM to return a Questions object
create one ground truth record for each generated question
"""

from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [19]:
# trying for 5 docs:

from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [20]:
"""
Parallel processing:
Running the calls one after another wastes most of the time waiting on the network. 
Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. 
"""

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# This submits one job per document, updates the progress bar when a job finishes, and collects the results.

In [21]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [22]:
# splitting into 2 lists:

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [23]:
"""
With 5 questions per document, you should get roughly 5x the number of documents.

Calculate the total cost:
"""

from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08782800000000003

In [25]:
# We'll calculate total cost several times in this module, so the utility file has a helper for it:

from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08782800000000003

In [26]:
# Create a dataframe so we can look at the records as a table and save them as a CSV file.

import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [28]:
"""
Because we generated the questions from specific documents, we know which document is correct for each question. 
We now have the ground truth we need for evaluation.
"""

df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [29]:
"""
Search Evaluation:
For each question in our ground truth dataset, we run search. 
Then we check whether the results include the correct document.
"""

import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [30]:
# Load the documents and build a minsearch index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [31]:
"""
Wrap the search call in a function called text_search. The name is deliberate. 
Later we'll write vector_search or a hybrid version and run the exact same evaluation on it. 
Everything downstream only needs a function that takes a query and returns results, so we can swap one for another. 
That mirrors how RAG works: the retrieval step doesn't care which search function sits behind it.
"""

def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
"""
Collecting relevance data:

"""